# 02 — Jakość danych: Uzupełnianie luk i walidacja

Ten notebook kontynuuje pracę z `01_data_acquisition` i obejmuje:

1. **Wczytanie przetworzonych danych** z poprzedniego kroku
2. **Szczegółowa analiza luk** — identyfikacja wszystkich okresów brakujących danych
3. **Uzupełnianie luk** — interpolacja dla krótkich luk, korelacja dla dłuższych
4. **Detekcja wartości odstających** — oznaczenie podejrzanych wartości
5. **Walidacja i eksport** — czysty zbiór danych do analizy statystycznej

---

## Konfiguracja

**Prompt do LLM:**
> *"Napisz kod konfiguracji notebooka: załaduj moduły src/imgw (load_processed, save_processed) i src/data_quality (check_completeness, find_gaps, fill_gaps_interpolation, fill_gaps_correlation, detect_outliers)."*

**Użyte moduły/funkcje:** `src.imgw`: load_processed, save_processed; `src.data_quality`: check_completeness, find_gaps, fill_gaps_interpolation, fill_gaps_correlation, detect_outliers

In [2]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from src.imgw import load_processed, save_processed
from src.data_quality import (
    check_completeness,
    find_gaps,
    fill_gaps_interpolation,
    fill_gaps_correlation,
    detect_outliers,
)

pd.set_option('display.max_columns', 15)
print('Moduły załadowane.')

Moduły załadowane.


### Moduł `src/data_quality.py` — kontrola jakosci i uzupelnianie danych

**Prompt do LLM tworzący ten moduł:**
> *"Napisz modul Pythona src/data_quality.py do kontroli jakosci danych hydrologicznych.
> Moduł powinien:
> 1. Sprawdzać kompletność danych per stacja i rok (ile dni ma dane, ile brakuje).
> 2. Znajdować luki — ciągłe okresy brakujących danych, zwracać start/koniec/długość.
> 3. Uzupełniać krótkie luki (do N dni) interpolacją liniową.
> 4. Uzupełniać dłuższe luki metodą korelacji ze stacją referencyjną (regresją liniową).
> 5. Wykrywać wartości odstające z-score na danych log-transformowanych.
> Dane wejsciowe: DataFrame z kolumnami station_id, date, discharge_m3s, water_level_cm."*

**Wynik:** Moduł `src/data_quality.py` z funkcjami:
- `check_completeness(df, station_id)` — statystyki kompletności per rok
- `find_gaps(df, station_id, column)` — lista luk z czasem trwania
- `fill_gaps_interpolation(df, station_id, column, max_gap_days)` — interpolacja liniowa
- `fill_gaps_correlation(df, target_id, reference_id, column)` — regresja między stacjami
- `detect_outliers(df, station_id, column, z_threshold)` — detekcja outlierów

**Prompt do LLM:**
> *"Napisz kod ktory użyje funkcji load_processed() z modułu src/imgw do wczytania danych z pliku Parquet ../data/processed/daily_hydro.parquet. Wyświetl liczbę wierszy, listę stacji i zakres dat."*

**Użyte moduły/funkcje:** `src.imgw.load_processed()` — odczyt Parquet z zachowanymi typami

In [3]:
# Wczytaj dane z kroku 01
df = load_processed('../data/processed/daily_hydro.parquet')

print(f'Wczytano {len(df)} wierszy')
print(f'Stacje: {df["station_id"].unique().tolist()}')
print(f'Zakres dat: {df["date"].min()} — {df["date"].max()}')
df.head()

Wczytano 31410 wierszy
Stacje: ['151160150', '151160170']
Zakres dat: 1980-11-01 00:00:00 — 2023-10-31 00:00:00


,station_id,station_name,river_name,water_level_cm,discharge_m3s,date
0,151160150,MALCZYCE,Odra (1),110.0,136.0,1980-11-01
1,151160150,MALCZYCE,Odra (1),110.0,136.0,1980-11-02
2,151160150,MALCZYCE,Odra (1),112.0,138.0,1980-11-03
3,151160150,MALCZYCE,Odra (1),112.0,138.0,1980-11-04
4,151160150,MALCZYCE,Odra (1),112.0,138.0,1980-11-05


---
## Krok 1: Szczegółowa analiza luk

Zidentyfikujmy wszystkie okresy brakujących danych i oceńmy ich istotność.

**Prompt do LLM:**
> *"Napisz kod ktory użyje funkcji find_gaps() z modułu src/data_quality i pogrupuje znalezione luki wg kategorii długości używając pandas cut(). Pokaż liczbę i łączny czas luk w każdej kategorii per stacja."*

**Użyte moduły/funkcje:** `src.data_quality.find_gaps()` + `pandas.cut()` — kategoryzacja luk

In [ ]:
# Analiza luk dla każdej stacji
for sid in df['station_id'].unique():
    name = df.loc[df['station_id'] == sid, 'station_name'].iloc[0]
    gaps = find_gaps(df, sid, 'discharge_m3s')

    print(f'\n=== {name} ({sid}) ===')
    if gaps.empty:
        print('  Brak luk — kompletna seria!')
        continue

    print(f'  Łączna liczba luk: {len(gaps)}')
    print(f'  Łączna liczba brakujących dni: {gaps["duration_days"].sum()}')
    print(f'  Rozkład wielkości luk:')
    bins = [0, 1, 3, 7, 30, 365, 99999]
    labels = ['1 dzień', '2-3 dni', '4-7 dni', '8-30 dni', '31-365 dni', '>1 rok']
    gaps['category'] = pd.cut(gaps['duration_days'], bins=bins, labels=labels)
    print(gaps.groupby('category', observed=True)['duration_days'].agg(['count', 'sum']).to_string())

---
## Krok 2: Uzupełnianie krótkich luk (interpolacja)

Dla luk **≤5 dni** interpolacja liniowa jest rozsądnym podejściem dla dobowych danych przepływu.
Dłuższe luki wymagają innych metod (korelacja ze stacją referencyjną, krzywe regionalne itp.).

**Prompt do LLM:**
> *"Napisz kod ktory użyje funkcji fill_gaps_interpolation() z modułu src/data_quality do uzupełnienia luk <= 5 dni w kolumnie discharge_m3s. Dla kazdej stacji pokaż ile wartości było brakujących przed i po."*

**Użyte moduły/funkcje:** `src.data_quality.fill_gaps_interpolation()` — pandas interpolate() z limitem

In [ ]:
# Uzupełnij krótkie luki interpolacją liniową (maks. 5 dni)
df_filled = df.copy()
for sid in df['station_id'].unique():
    name = df.loc[df['station_id'] == sid, 'station_name'].iloc[0]

    before = df.loc[df['station_id'] == sid, 'discharge_m3s'].isna().sum()
    df_filled = fill_gaps_interpolation(df_filled, sid, 'discharge_m3s', max_gap_days=5)
    after = df_filled.loc[df_filled['station_id'] == sid, 'discharge_m3s'].isna().sum()

    print(f'{name}: {before} → {after} brakujących wartości (uzupełniono {before - after})')

---
## Krok 3: Uzupełnianie dłuższych luk (korelacja stacji)

Jeśli masz **dwie stacje na tej samej rzece**, możesz użyć regresji liniowej między nimi do uzupełnienia większych luk.

Ten krok wymaga co najmniej 2 stacji. Pomiń jeśli masz tylko jedną.

Metoda dopasowuje: `Q_cel = a × Q_referencyjna + b` na danych nakładających się, a następnie estymuje brakujące wartości.

**Prompt do LLM:**
> *"Napisz kod ktory użyje funkcji fill_gaps_correlation() z modułu src/data_quality do uzupełnienia luk w stacji docelowej na podstawie stacji referencyjnej. Użyj pierwszej stacji jako cel, drugiej jako referencja. Pokaż R2 korelacji i ile wartości uzupełniono."*

**Użyte moduły/funkcje:** `src.data_quality.fill_gaps_correlation()` — scipy.stats.linregress + predykcja brakujacych

In [ ]:
station_ids = df_filled['station_id'].unique()

if len(station_ids) >= 2:
    target_id = station_ids[0]
    reference_id = station_ids[1]

    target_name = df_filled.loc[df_filled['station_id'] == target_id, 'station_name'].iloc[0]
    ref_name = df_filled.loc[df_filled['station_id'] == reference_id, 'station_name'].iloc[0]

    print(f'Uzupełnianie luk w {target_name} na podstawie {ref_name}:')
    df_filled = fill_gaps_correlation(df_filled, target_id, reference_id, 'discharge_m3s')

    # Sprawdź pozostałe luki
    remaining = df_filled.loc[df_filled['station_id'] == target_id, 'discharge_m3s'].isna().sum()
    print(f'Pozostałe brakujące wartości: {remaining}')
else:
    print('Tylko jedna stacja — pomijam uzupełnianie korelacyjne.')

**Prompt do LLM:**
> *"Napisz kod ktory połączy dane przepływu z dwóch stacji po dacie i narysuje wykres scatter z plotly express z trendline='ols'. Dodaj etykiety osi z nazwami stacji."*

**Użyte moduły/funkcje:** `plotly.express.scatter()` z `trendline='ols'`

In [ ]:
# Wykres korelacji stacji (jeśli 2+ stacje)
if len(station_ids) >= 2:
    t = df_filled[df_filled['station_id'] == station_ids[0]][['date', 'discharge_m3s']].set_index('date')
    r = df_filled[df_filled['station_id'] == station_ids[1]][['date', 'discharge_m3s']].set_index('date')
    t.columns = ['Q_target']
    r.columns = ['Q_reference']
    merged = t.join(r).dropna()

    t_name = df_filled.loc[df_filled['station_id'] == station_ids[0], 'station_name'].iloc[0]
    r_name = df_filled.loc[df_filled['station_id'] == station_ids[1], 'station_name'].iloc[0]

    fig = px.scatter(
        merged, x='Q_reference', y='Q_target',
        title=f'Korelacja stacji: {t_name} vs {r_name}',
        labels={'Q_reference': f'Q {r_name} [m³/s]', 'Q_target': f'Q {t_name} [m³/s]'},
        trendline='ols',
        opacity=0.3,
        height=500,
    )
    fig.show()

---
## Krok 4: Detekcja wartości odstających

Wykrywanie ekstremalnych wartości za pomocą z-score na danych przepływu po transformacji logarytmicznej (przepływ ma zazwyczaj rozkład log-normalny).

**Prompt do LLM:**
> *"Napisz kod ktory użyje funkcji detect_outliers() z modułu src/data_quality do wykrycia wartości odstających w discharge_m3s. Prog z_threshold=4.0. Wyświetl datę i wartość każdego outliera."*

**Użyte moduły/funkcje:** `src.data_quality.detect_outliers()` — np.log1p + z-score, prog 4.0

In [ ]:
for sid in df_filled['station_id'].unique():
    name = df_filled.loc[df_filled['station_id'] == sid, 'station_name'].iloc[0]
    outliers = detect_outliers(df_filled, sid, 'discharge_m3s', z_threshold=4.0)

    print(f'\n=== {name} ({sid}): {len(outliers)} potencjalnych wartości odstających ===')
    if not outliers.empty:
        print(outliers[['date', 'discharge_m3s']].to_string(index=False))

---
## Krok 5: Porównanie przed/po

**Prompt do LLM:**
> *"Napisz kod ktory narysuje wykres porównawczy używając plotly graph_objects: dwa trace Scatter na jednym wykresie — dane oryginalne i uzupelnione. Osobny wykres dla każdej stacji."*

**Użyte moduły/funkcje:** `plotly.graph_objects.Scatter()` — dwa trace na jednym wykresie

In [ ]:
# Porównanie danych oryginalnych i uzupełnionych
for sid in df_filled['station_id'].unique():
    name = df_filled.loc[df_filled['station_id'] == sid, 'station_name'].iloc[0]

    orig = df[df['station_id'] == sid][['date', 'discharge_m3s']].copy()
    filled = df_filled[df_filled['station_id'] == sid][['date', 'discharge_m3s']].copy()
    orig.columns = ['date', 'original']
    filled.columns = ['date', 'filled']
    comp = orig.merge(filled, on='date')

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=comp['date'], y=comp['original'], name='Oryginalne', opacity=0.7))
    fig.add_trace(go.Scatter(x=comp['date'], y=comp['filled'], name='Uzupełnione', opacity=0.7))
    fig.update_layout(
        title=f'Przepływ: oryginalne vs uzupełnione — {name}',
        xaxis_title='Data', yaxis_title='Q [m³/s]',
        hovermode='x unified', height=450,
    )
    fig.show()

---
## Krok 6: Zapis oczyszczonych danych

**Prompt do LLM:**
> *"Napisz kod ktory użyje funkcji save_processed() z modułu src/imgw do zapisania oczyszczonego DataFrame jako Parquet w ../data/processed/daily_hydro_clean.parquet."*

**Użyte moduły/funkcje:** `src.imgw.save_processed()` — zapis gotowy do analizy w notebook 03

In [ ]:
# Zapisz oczyszczony/uzupełniony zbiór danych
save_processed(df_filled, '../data/processed/daily_hydro_clean.parquet')
print('Oczyszczone dane zapisane. Gotowe do analizy statystycznej w notebooku 03.')

---
## Podsumowanie

W tym notebooku:
1. Przeanalizowaliśmy luki danych wg kategorii wielkości
2. Uzupełniliśmy krótkie luki (≤5 dni) interpolacją liniową
3. Uzupełniliśmy dłuższe luki metodą korelacji między stacjami
4. Wykryliśmy potencjalne wartości odstające
5. Porównaliśmy dane oryginalne z uzupełnionymi

**Dalej:** `03_hydro_statistics.ipynb` — krzywe uporządkowane przepływu, przepływy charakterystyczne, wzorce miesięczne/roczne